In [ ]:
#!/usr/bin/env python3
"""
NB10: Health Economics (QALY + ICER)
Evaluates the real economic impact of the AED scenarios.
- Survival Curve: S(t) = exp(-0.11 * t) [Valenzuela et al. 1997 / standard decay]
- QALY: Quality-Adjusted Life Years saved. Assume avg 10 QALYs per survivor.
- ICER: Incremental Cost-Effectiveness Ratio (Cost / QALY).
"""



# NB 10 — Health Economics (QALY & ICER)

Transforms spatial coverage improvements into real-world health outcomes.
We apply a standard cardiac arrest survival decay function ($\sim$10% drop per minute)
to estimate the expected number of survivors under baseline and expansion scenarios.
Then, we compute Quality-Adjusted Life Years (QALY) and the 
Incremental Cost-Effectiveness Ratio (ICER).

The WHO considers interventions highly cost-effective if ICER < 1x GDP/capita,
and cost-effective if < 3x GDP/capita. For Belgium, this threshold is roughly €120,000/QALY.

**Expected runtime: ~1 minute**


In [ ]:
Setup
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.')
V3_DIR   = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR  = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
FIG_DIR  = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:
Load Data
mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
scenarios = pd.read_csv(FIG_DIR / 'table_05_scenarios.csv')

# Only consider likely OHCA cases for the QALY analysis
# E.g. P033 Trauma, P010 Respiratory, P026 Unclear are the high-acuity ones 
# Let's subset to ~10% of missions that represent actual OHCAs based on literature
N_CARDIAC_ARRESTS = int(len(mission) * 0.05) # 5% assumed OHCA rate
print(f"Assuming {N_CARDIAC_ARRESTS} true OHCA events in the dataset.")

# Function for survival
def prob_survival(t_minutes):
    """Valenzuela exponential decay model"""
    # Assuming baseline survival at t=0 is 60%, decaying ~10% rel per minute
    # S(t) = 0.60 * exp(-0.11 * t)
    return np.clip(0.60 * np.exp(-0.11 * t_minutes), 0, 1.0)

# Simulate OHCA cohort response times (sample from the actual response time distribution)
ohca_rt = mission['response_min'].dropna().sample(N_CARDIAC_ARRESTS, random_state=42).values

# Baseline expected survivors
base_survivors = np.sum(prob_survival(ohca_rt))
print(f"Baseline Expected Survivors: {base_survivors:.1f} out of {N_CARDIAC_ARRESTS}")




In [ ]:
Evaluate Scenarios
# We approximate the reduction in response time based on the delta in median distance
base_med_dist = mission['dist_to_aed_km'].median()
base_med_rt = mission['response_min'].median()

# A very rough approximation: every 1km reduction in distance to AED saves 3 minutes
# if the AED is used by a bystander.
# However, bystander deployment rate is low (~15%).
BYSTANDER_USE_RATE = 0.15

results = []
prev_cost = 0
prev_surv = base_survivors

for i, row in scenarios.iterrows():
    n_aeds = row['n_new_aed']
    med_dist = row['median_dist_km']
    cost = row['total_cost_eur']
    
    # Delta distance (how much closer the median AED is)
    dist_saved = base_med_dist - med_dist
    
    # Time saved if an AED is nearby (cap at 0, only applied to the bystander subset)
    # Assume 1 km walking/running = 6 minutes. Round trip = 12 minutes. 
    # That's why distance reduction matters highly to time.
    time_saved_mins = max(0, dist_saved * 12)
    
    # Apply time savings only to the bystander subset
    new_rt = ohca_rt.copy()
    bystander_mask = np.random.rand(N_CARDIAC_ARRESTS) < BYSTANDER_USE_RATE
    new_rt[bystander_mask] = np.clip(new_rt[bystander_mask] - time_saved_mins, 0.5, None)
    
    survivors = np.sum(prob_survival(new_rt))
    survs_saved = survivors - base_survivors
    
    # Health Econ
    # Assume 1 survivor = 10 QALYs (Quality-Adjusted Life Years)
    qalys = survs_saved * 10
    
    # Incremental calculations
    delta_cost = cost - prev_cost
    delta_surv = survivors - prev_surv
    icer = delta_cost / (delta_surv * 10) if delta_surv > 0 else np.nan
    
    results.append({
        'Scenario': row['scenario'],
        'Added_AEDs': n_aeds,
        'Extra_Survivors': survivors - base_survivors,
        'QALYs_Saved': qalys,
        'Total_Cost_EUR': cost,
        'ICER_EUR_per_QALY': icer
    })
    
    prev_cost = cost
    prev_surv = survivors

df_he = pd.DataFrame(results)
print(df_he.to_string())
df_he.to_csv(FIG_DIR / 'table_10_health_economics.csv', index=False)




In [ ]:
Figure: ICER Curve
print("Rendering Fig 7 (Health Economics)...")
fig7, ax = plt.subplots(figsize=(8, 6), dpi=200)

ax.plot(df_he['Added_AEDs'], df_he['ICER_EUR_per_QALY']/1000, marker='o', lw=2, color='#8e44ad')

# WHO Threshold
WHO_THRESH_K = 120
ax.axhline(WHO_THRESH_K, color='red', ls='--', label=f'WHO Threshold (~€{WHO_THRESH_K}k/QALY)')

ax.set_title('(f) Incremental Cost-Effectiveness Ratio (ICER)\nAssuming 15% bystander utilization rate', fontweight='bold')
ax.set_xlabel('New AEDs Deployed')
ax.set_ylabel('Incremental Cost per QALY Saved [kEUR]')
ax.legend()
ax.grid(ls=':', alpha=0.6)

# Annotate
if df_he['ICER_EUR_per_QALY'].max() > WHO_THRESH_K * 1000:
    idx = df_he['ICER_EUR_per_QALY'] > WHO_THRESH_K * 1000
    first_ineff = df_he[idx].iloc[0]
    ax.annotate(f"Crosses Threshold at S{int(first_ineff['Added_AEDs'])}",
                xy=(first_ineff['Added_AEDs'], first_ineff['ICER_EUR_per_QALY']/1000),
                xytext=(0, 20), textcoords='offset points', arrowprops=dict(arrowstyle="->"), ha='center')
else:
    ax.text(0.5, 0.8, "All scenarios technically cost-effective\nunder strict WHO threshold, but\ndiminishing returns are steep.",
            transform=ax.transAxes, ha='center', bbox=dict(facecolor='#fdfefe', edgecolor='#aaa', boxstyle='round,pad=0.5'))

fig7.tight_layout()
fig7.savefig(FIG_DIR / 'fig7_icer_curve.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig7)
print("  -> saved fig7_icer_curve.png")
print("\n=== NB10 Complete ===")

